# OpenVINO Physical AI APIs to test a trained policy

### Test a previously trained Pi0.5 policy without a physical robot, then render the final rollout in MuJoCo

<img src="media/4.Simulation-only.png" width="800">

## 1) Install OpenVINO Physical AI

In [ ]:
from pathlib import Path
import subprocess
import sys

WORKSPACE = Path.cwd().resolve()
TUTORIALS_REF = "docs/tutorials"
TUTORIALS_REPO = "https://github.com/openvinotoolkit/physicalai.git"
TUTORIAL_REPO_DIR = WORKSPACE / "_physicalai_tutorial_repo"

local_requirements = WORKSPACE / "requirements.txt"
local_helper = WORKSPACE / "physicalai_pi05_helper.py"

if not local_requirements.exists() or not local_helper.exists():
    if not TUTORIAL_REPO_DIR.exists():
        subprocess.check_call([
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            TUTORIALS_REF,
            TUTORIALS_REPO,
            str(TUTORIAL_REPO_DIR),
        ])
    TUTORIAL_SUPPORT_DIR = TUTORIAL_REPO_DIR / "examples" / "tutorials"
else:
    TUTORIAL_SUPPORT_DIR = WORKSPACE

requirements_file = local_requirements if local_requirements.exists() else TUTORIAL_SUPPORT_DIR / "requirements.txt"
HELPER_DIR = local_helper.parent if local_helper.exists() else TUTORIAL_SUPPORT_DIR

PHYSICALAI_VERSION = "0.1.1"

%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu -r {requirements_file}
%pip install -q physicalai=={PHYSICALAI_VERSION}
%pip install -q "mujoco>=3.2.0" imageio

## 2) Configure the Notebook

This notebook validates a trained and OpenVINO-optimized Pi0.5 policy package without connecting to a physical robot.

By default, the model package is downloaded from the official [OpenVINO Physical AI Hugging Face collection](https://huggingface.co/collections/OpenVINO/physical-ai). The model is converted from `lerobot/pi05_libero_finetuned_v044`, whose model card references the LeRobot-format [`HuggingFaceVLA/libero`](https://huggingface.co/datasets/HuggingFaceVLA/libero) dataset.

The notebook downloads only the metadata, one episode parquet, and the corresponding videos needed for replay validation. This keeps the validation path lightweight while still showing model loading, policy inference, replay comparison, and the final MuJoCo rollout visualization.

Useful environment variables:
- `PHYSICALAI_PI05_EXPORT_DIR`
- `PHYSICALAI_PI05_REPLAY_DATASET`
- `PHYSICALAI_PI05_REPLAY_EPISODE`
- `PHYSICALAI_PI05_TASK`
- `PHYSICALAI_ASSETS_DIR`

In [ ]:
# ============================================================
# USER PARAMETERS - Edit these before running if needed.
# ============================================================

MODEL_REPO_ID = "OpenVINO/pi05-libero-fp16-ov"
DATASET_REPO_ID = "HuggingFaceVLA/libero"
DATASET_NAME = ""

from pathlib import Path
import os
import sys

import openvino as ov
import openvino_tokenizers  # Registers tokenizer custom ops used by tokenizer.xml.
from IPython.display import display
from physicalai.inference import InferenceModel

WORKSPACE = Path.cwd().resolve()
if "HELPER_DIR" not in globals():
    HELPER_DIR = WORKSPACE
sys.path.append(str(Path(HELPER_DIR).resolve()))
sys.path.append(str((WORKSPACE / "notebooks").resolve()))

from physicalai_pi05_helper import (  # noqa: E402
    benchmark_with_fallback,
    download_pi05_package,
    openvino_config_for_device,
    prepare_replay_episode,
    run_mujoco_visualization,
    run_replay_visualization,
)

TASK_TEXT = os.environ.get("PHYSICALAI_PI05_TASK")
ASSETS_DIR = Path(os.environ.get("PHYSICALAI_ASSETS_DIR", WORKSPACE / "physicalai_assets")).expanduser().resolve()

MODEL_DIR = Path(os.environ["PHYSICALAI_PI05_EXPORT_DIR"]).expanduser().resolve() if os.environ.get("PHYSICALAI_PI05_EXPORT_DIR") else None
REPLAY_DATASET_DIR = Path(os.environ["PHYSICALAI_PI05_REPLAY_DATASET"]).expanduser().resolve() if os.environ.get("PHYSICALAI_PI05_REPLAY_DATASET") else None
REPLAY_EPISODE_ID = int(os.environ.get("PHYSICALAI_PI05_REPLAY_EPISODE", "0"))
RUN_REPLAY_VALIDATION = True

CACHE_DIR = ASSETS_DIR / "cache" / "pi05_openvino"
VIS_DIR = ASSETS_DIR / "visualizations" / "pi05_without_robot"

core = ov.Core()
print("OpenVINO:", ov.__version__)
print("Available devices:", core.available_devices)
print("Replay dataset:", DATASET_REPO_ID or REPLAY_DATASET_DIR)
print("Replay validation:", "enabled" if RUN_REPLAY_VALIDATION else "disabled")

## 3) Download the Pi0.5 OpenVINO Policy Package

A PhysicalAI policy package contains the OpenVINO intermediate representation files (`pi05.xml`, `pi05.bin`), tokenizer artifacts, `manifest.json`, and processor metadata.

In [ ]:
MODEL_DIR = download_pi05_package(MODEL_REPO_ID, ASSETS_DIR, MODEL_DIR)
print("[DONE] PhysicalAI OpenVINO package is ready.")

## 4) Download and Prepare One LIBERO Replay Episode

For a fast validation pass, the notebook downloads dataset metadata, one episode parquet, and the corresponding camera videos from `HuggingFaceVLA/libero`. This is enough to build the same observation dictionary shape used by the Pi0.5 OpenVINO package.

In [ ]:
replay = None
if RUN_REPLAY_VALIDATION:
    replay = prepare_replay_episode(
        repo_id=DATASET_REPO_ID,
        dataset_name=DATASET_NAME,
        assets_dir=ASSETS_DIR,
        episode_id=REPLAY_EPISODE_ID,
        dataset_dir=REPLAY_DATASET_DIR,
    )
    if TASK_TEXT is None:
        TASK_TEXT = replay.task or "pick up the object"
    print(f"[DONE] Replay episode {replay.episode_id}: {len(replay.episode_df)} frames at {replay.fps:.1f} FPS")
    print("[DONE] Replay image keys:", replay.image_keys)
    print("[DONE] Task:", TASK_TEXT)
else:
    TASK_TEXT = TASK_TEXT or "pick up the object"
    print("[SKIP] Replay validation disabled. The final MuJoCo cell will render a scripted pick-and-place trajectory.")

## 5) Select an OpenVINO Device


In [ ]:
import ipywidgets as widgets

device_options = list(core.available_devices)
default_device = "GPU" if "GPU" in device_options else "CPU"
TARGET_DEVICE = widgets.Dropdown(
    options=device_options,
    value=default_device if default_device in device_options else device_options[0],
    description="Device:",
)
display(TARGET_DEVICE)


## 6) Load and Benchmark with PhysicalAI Runtime

`InferenceModel.load()` reads the PhysicalAI policy package, selects the OpenVINO backend, applies preprocessors/postprocessors from the manifest, and exposes `predict_action_chunk()` plus `select_action()` for deployment.

When a replay dataset is configured, this cell benchmarks the policy on one replay observation. Otherwise it validates that the OpenVINO policy package can be loaded on the selected device, with CPU fallback.

In [ ]:
selected_result = {"device": TARGET_DEVICE.value}

if replay is not None:
    physicalai_model, selected_result = benchmark_with_fallback(
        model_dir=MODEL_DIR,
        replay=replay,
        task=TASK_TEXT,
        device=TARGET_DEVICE.value,
        cache_dir=CACHE_DIR,
        runs=5,
    )

    print("[RESULT] PhysicalAI Pi0.5 OpenVINO deployment")
    for key, value in selected_result.items():
        print(f"  {key}: {value}")
else:
    try:
        physicalai_model = InferenceModel.load(
            MODEL_DIR,
            backend="openvino",
            device=TARGET_DEVICE.value,
            **openvino_config_for_device(CACHE_DIR),
        )
    except RuntimeError as exc:
        if TARGET_DEVICE.value == "CPU":
            raise
        print(f"[WARN] OpenVINO load failed on {TARGET_DEVICE.value}: {exc}")
        print("[INFO] Falling back to CPU so the notebook can continue.")
        physicalai_model = InferenceModel.load(
            MODEL_DIR,
            backend="openvino",
            device="CPU",
            **openvino_config_for_device(CACHE_DIR),
        )
        selected_result = {"device": "CPU"}
    print(f"[DONE] Loaded PhysicalAI Pi0.5 OpenVINO package on {selected_result['device']}")

## 7) Optional Replay Visualization

When replay validation is enabled, this cell compares Pi0.5 OpenVINO actions with recorded expert actions while showing both camera views. The MAE is a domain-match signal, not an OpenVINO numerical correctness test.

In [ ]:
replay_result = None
if replay is not None:
    replay_result = run_replay_visualization(
        model=physicalai_model,
        model_dir=MODEL_DIR,
        replay=replay,
        task=TASK_TEXT,
        device=selected_result["device"],
        cache_dir=CACHE_DIR,
        output_dir=VIS_DIR,
        max_rendered_frames=120,
        render_stride=3,
    )

    print("[RESULT] Replay steps:", replay_result["steps"])
    print("[RESULT] Rendered frames:", replay_result["rendered_frames"])
    print("[RESULT] Avg select_action latency ms:", replay_result["avg_select_action_ms"])
    print("[RESULT] Avg MAE vs expert action:", replay_result["avg_mae"])
    print("[RESULT] Per-joint MAE:", replay_result["per_joint_mae"])
    print("[INTERPRETATION]", replay_result["interpretation"])
    print("[DONE] Saved replay GIF:", replay_result["gif_path"])
    display(replay_result["gif"])
else:
    print("[SKIP] Replay visualization skipped because no replay dataset is configured.")

## 8) MuJoCo Pick-and-Place Visualization

This final cell renders a lightweight MuJoCo pick-and-place scene so users can see a robotless simulation result. The official OpenVINO model is converted from a LIBERO/Panda policy, while this notebook uses a compact built-in MuJoCo scene for fast visualization. Because the LIBERO action space is not the same as this toy arm's joint-control space, the MuJoCo scene uses a scripted pick-and-place trajectory instead of directly driving the arm with the model's predicted action tensor.

Use the replay section above for actual OpenVINO policy inference, latency, and action-vs-expert validation. Use this MuJoCo section as a visual smoke test that the notebook can produce a simulation-style result without a physical robot.

In [ ]:
mujoco_result = run_mujoco_visualization(
    actions=None,
    output_dir=VIS_DIR,
    source="scripted MuJoCo pick-and-place trajectory",
    max_rendered_frames=180,
)

print("[RESULT] MuJoCo frames:", mujoco_result["frames"])
print("[RESULT] MuJoCo source:", mujoco_result["source"])
print("[DONE] Saved MuJoCo GIF:", mujoco_result["gif_path"])
display(mujoco_result["gif"])
